# Part 1 · Dataset Exploration & Preparation
**Owner: Teddy**

This section explores all three data sources for the EcoSort assistant and builds the preprocessing pipelines the other parts depend on:

1. **RealWaste images** (9 categories) — for the CNN (Part 2)
2. **waste_descriptions.csv** (5,000 text descriptions) — for the text classifier (Part 3)
3. **waste_policy_documents.json** (14 policy docs) — for the RAG system (Part 4)


### Setup (VS Code / local) — run first
Unzips `data/realwaste.zip` into `data/RealWaste/` the first time (fast on your local disk).

In [ ]:
import os, zipfile, pathlib, shutil
if not pathlib.Path('data/RealWaste').exists() and pathlib.Path('data/realwaste.zip').exists():
    with zipfile.ZipFile('data/realwaste.zip') as z: z.extractall('data/_tmp')
    for p in pathlib.Path('data/_tmp').rglob('RealWaste'):
        shutil.move(str(p), 'data/RealWaste'); break
    shutil.rmtree('data/_tmp', ignore_errors=True)
IMAGE_DIR = 'data/RealWaste'
print('Images present:', os.path.isdir(IMAGE_DIR))

In [ ]:
import os, re, json, random, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
sns.set_theme(style='whitegrid')
np.random.seed(42); random.seed(42)
IMG_HEIGHT = IMG_WIDTH = 224   # target size for the CNN (Part 2)
BATCH_SIZE = 32
# Canonical category order — derived from the folder names so EVERY part uses the same label order
CATEGORIES = sorted([d.name for d in pathlib.Path(IMAGE_DIR).glob('*') if d.is_dir()])
print('Categories:', CATEGORIES)

## 1.1 Load and Explore the RealWaste Image Dataset

In [ ]:
# Class distribution
counts = {c: len(list((pathlib.Path(IMAGE_DIR)/c).glob('*'))) for c in CATEGORIES}
total = sum(counts.values())
print('Total images:', total)
s = pd.Series(counts).sort_values()
ax = s.plot.barh(figsize=(8,4), color='#2a9d8f')
ax.set_title(f'RealWaste: images per category (n={total})'); ax.set_xlabel('count')
for i,v in enumerate(s): ax.text(v+5, i, str(v), va='center')
plt.tight_layout(); plt.show()
imbalance = max(counts.values())/min(counts.values())
print(f'Most: {s.idxmax()} ({s.max()}) | Least: {s.idxmin()} ({s.min()}) | Imbalance ratio: {imbalance:.2f}x')

In [ ]:
# Image characteristics: resolution, colour mode, and integrity check (sample for speed)
sizes, modes, corrupt = [], {}, []
sample_per_class = 60
for c in CATEGORIES:
    files = list((pathlib.Path(IMAGE_DIR)/c).glob('*'))
    for f in random.sample(files, min(sample_per_class, len(files))):
        try:
            with Image.open(f) as im:
                sizes.append(im.size); modes[im.mode] = modes.get(im.mode,0)+1
        except Exception as e:
            corrupt.append((str(f), str(e)))
sz = pd.DataFrame(sizes, columns=['w','h'])
print('Sampled', len(sz), 'images')
print('Unique resolutions:', sz.value_counts().head().to_dict())
print('Colour modes:', modes)
print('Corrupt/unreadable:', len(corrupt))

In [ ]:
# One sample image per category
fig, ax = plt.subplots(3, 3, figsize=(10,10))
for a, c in zip(ax.ravel(), CATEGORIES):
    f = next((pathlib.Path(IMAGE_DIR)/c).glob('*'))
    a.imshow(Image.open(f)); a.set_title(c); a.axis('off')
plt.tight_layout(); plt.show()

**Image findings**

- **4,752 images across 9 balanced-in-schema but imbalanced-in-count categories.** The largest class (**Plastic, 921**) has ~**2.9×** the images of the smallest (**Textile Trash, 318**). This imbalance is addressed in Part 2 with **class weights** and **augmentation**, and we report **macro-averaged** metrics, not just accuracy.
- **Every image is exactly 524×524 and RGB** — no grayscale, no corrupt files, no mixed resolutions. So preprocessing is simple and safe: a single **resize to 224×224** (MobileNetV2's native size) with no aspect-ratio or channel handling needed.
- **Challenge — visually similar classes:** Paper vs Cardboard, Food Organics vs Vegetation, and the different Glass items share colour/texture, so we expect these to be the main confusions in the CNN's confusion matrix.
- **Challenge — controlled capture:** images are shot on a plain conveyor-style background with even lighting. A model trained on them may not transfer perfectly to cluttered real resident phone photos (noted in Limitations).

## 1.2 Explore the Text Descriptions (waste_descriptions.csv)

In [ ]:
desc = pd.read_csv('data/waste_descriptions.csv')
print('shape:', desc.shape)
print('columns:', list(desc.columns))
desc.head()

In [ ]:
# Category balance (should mirror the image categories)
vc = desc['category'].value_counts()
ax = vc.sort_values().plot.barh(figsize=(8,4), color='#e76f51')
ax.set_title('Descriptions per category'); plt.tight_layout(); plt.show()
print('Same 9 categories as images:', sorted(desc['category'].unique()) == CATEGORIES)

In [ ]:
# Description length & vocabulary
desc['n_words'] = desc['description'].str.split().str.len()
print('Words per description  min/mean/max:', desc['n_words'].min(), round(desc['n_words'].mean(),1), desc['n_words'].max())
desc['n_words'].plot.hist(bins=range(0,12), figsize=(7,3), color='#264653')
plt.title('Description length (words)'); plt.xlabel('words'); plt.tight_layout(); plt.show()
from collections import Counter
vocab = Counter()
for d in desc['description'].str.lower().str.replace(r'[^a-z ]',' ', regex=True): vocab.update(d.split())
print('Vocabulary size:', len(vocab))
print('Top 12 words:', vocab.most_common(12))

In [ ]:
# Data-quality checks
print('Missing values per column:\n', desc.isna().sum())
print('\nDuplicate full rows      :', desc.duplicated().sum())
print('Duplicate descriptions    :', desc.duplicated(subset=['description']).sum())

### 1.2a Label-leakage check (important)
Before using the other columns as features we check how they relate to the label. If a column is (almost) a deterministic function of the category, training on it is **cheating** — the model would score ~100% but learn nothing about the actual text.

In [ ]:
for col in ['disposal_instruction', 'material_composition']:
    per_cat = desc.groupby('category')[col].nunique()
    print(f'{col}: unique values per category = {per_cat.unique()},  total unique = {desc[col].nunique()}')

**Text findings**

- **5,000 descriptions over the same 9 categories.** They are **very short and templated**: 1–10 words (mean ≈ 4.9) drawn from a tiny **309-word vocabulary** (e.g. *"medium sized empty glass bottle"*).
- **`material_composition` has exactly 1 unique value per category (9 total) and `disposal_instruction` only 5 per category** — both are essentially the label in disguise. **We therefore train the text classifier on the `description` column ONLY**, and use `disposal_instruction` only as reference text for the RAG system.
- **`common_confusion` is ~50% missing** (2,504 / 5,000 blank), so it is used for qualitative context, not as a feature.
- **61 duplicate descriptions** exist; we drop duplicates before splitting so the same text can't appear in both train and test (which would inflate the score).

## 1.2b Explore the Policy Documents (waste_policy_documents.json)

In [ ]:
with open('data/waste_policy_documents.json') as f: policies = json.load(f)
print('Documents:', len(policies))
print('Fields:', list(policies[0].keys()))
print('\nExample:', policies[0]['policy_type'], '->', policies[0]['categories_covered'])
print(policies[0]['document_text'][:300], '...')

In [ ]:
# How many policy documents cover each category? (RAG needs >=1 per category)
cov = Counter()
for p in policies:
    for c in p['categories_covered']: cov[c]+=1
pd.Series(cov).reindex(CATEGORIES).plot.bar(figsize=(8,3), color='#457b9d')
plt.title('Policy documents covering each category'); plt.ylabel('# docs'); plt.tight_layout(); plt.show()
lens = [len(p['document_text']) for p in policies]
print('Doc length (chars) min/mean/max:', min(lens), sum(lens)//len(lens), max(lens))

**Policy findings**

- **14 short policy documents** (≈571–951 characters each) with fields `policy_type`, `categories_covered`, `effective_date`, `jurisdiction`, `document_text`.
- **Every one of the 9 categories is covered by at least one document** (Vegetation/Metal/Misc by 4; **Paper by only 1**), so the RAG retriever can find relevant policy for any predicted category — though retrieval for **Paper** is the thinnest.
- Because documents are short, each can be treated as a **single retrieval chunk** (no long-document splitting needed).

## 1.3 Create the Data Pipelines
We build train/validation/test pipelines for each modality. **Split strategy: 70% train / 15% validation / 15% test**, stratified by category. 15% of ~4.7k images (~700) and of 5k texts (~750) is large enough for stable evaluation, while keeping enough training data for the minority classes.

### 1.3a Image pipeline (70/15/15)

In [ ]:
import tensorflow as tf
data_dir = pathlib.Path(IMAGE_DIR)
# First carve off 30% as a holdout (val+test), keep 70% for training
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.30, subset='training', seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode='categorical', class_names=CATEGORIES, shuffle=True)
holdout = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.30, subset='validation', seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode='categorical', class_names=CATEGORIES, shuffle=True)
# Split the 30% holdout evenly into validation (15%) and test (15%)
hb = tf.data.experimental.cardinality(holdout)
validation_ds = holdout.take(hb // 2)
test_dataset  = holdout.skip(hb // 2)
AUTOTUNE = tf.data.AUTOTUNE
train_ds      = train_ds.cache().prefetch(AUTOTUNE)
validation_ds = validation_ds.cache().prefetch(AUTOTUNE)
test_dataset  = test_dataset.cache().prefetch(AUTOTUNE)
for name, d in [('train',train_ds),('val',validation_ds),('test',test_dataset)]:
    print(name, 'batches:', tf.data.experimental.cardinality(d).numpy())

*Normalisation note:* we keep pixels as 0–255 here and apply the model-specific `preprocess_input` **inside** the CNN in Part 2, so the exact same scaling is guaranteed at train and inference time.

### 1.3b Text pipeline (clean → dedupe → stratified 70/15/15)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
STOP = ENGLISH_STOP_WORDS   # sklearn's built-in list (no external download needed)

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'[^a-z\s]', ' ', t)            # drop punctuation/digits
    tokens = [w for w in t.split() if w not in STOP]  # remove stopwords
    return ' '.join(tokens).strip()

# use ONLY description + category (avoid the leaky columns), drop duplicate texts
txt = desc[['description','category']].drop_duplicates(subset=['description']).reset_index(drop=True)
txt['clean'] = txt['description'].apply(clean_text)
print('After dedupe:', len(txt), 'rows')

train_txt, hold = train_test_split(txt, test_size=0.30, stratify=txt['category'], random_state=42)
val_txt, test_txt = train_test_split(hold, test_size=0.50, stratify=hold['category'], random_state=42)
print('text split -> train/val/test:', len(train_txt), len(val_txt), len(test_txt))

In [ ]:
# Demonstrate feature creation (TF-IDF) the way Part 3 will consume it
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(ngram_range=(1,2), min_df=2)
Xtr = vec.fit_transform(train_txt['clean'])
print('TF-IDF train matrix:', Xtr.shape, '(docs x features)')
print('Sample cleaned text:', train_txt['clean'].iloc[0])

### 1.3c RAG document preparation

In [ ]:
# Build the retrieval corpus: the 14 policy docs PLUS one distilled 'disposal tips' passage per category.
# (Embeddings + FAISS index are built in Part 4; here we just assemble & clean the text chunks.)
rag_corpus = []
for p in policies:
    rag_corpus.append({'source': p['policy_type'], 'categories': p['categories_covered'],
                       'text': ' '.join(p['document_text'].split())})
for cat, sub in desc.groupby('category'):
    tips = sorted(set(sub['disposal_instruction'].dropna()))[:5]
    rag_corpus.append({'source': f'{cat} disposal tips', 'categories': [cat],
                       'text': f'Disposal guidance for {cat}: ' + ' '.join(tips)})
print('RAG corpus chunks:', len(rag_corpus))
print('Example ->', rag_corpus[-1]['source'], '|', rag_corpus[-1]['text'][:120], '...')

In [ ]:
# Split summary
summary = pd.DataFrame({
    'modality': ['Images','Text','Policies (reference corpus)'],
    'total':    [sum(counts.values()), len(txt), len(rag_corpus)],
    'train':    ['70%', len(train_txt), 'n/a - retrieval reference'],
    'val':      ['15%', len(val_txt), 'n/a'],
    'test':     ['15%', len(test_txt), 'n/a']})
summary

## 1.4 Biases, Limitations & Data-Quality Notes

**Class imbalance (images).** 2.9× gap between Plastic (921) and Textile Trash (318). *Mitigation:* stratified splits, class weights + augmentation in the CNN, and macro-averaged metrics so minority classes count equally.

**Synthetic / templated text.** Descriptions are short (mean ≈ 5 words) and built from a 309-word vocabulary. A classifier can reach very high accuracy on them, but this may **overstate real-world performance** on free-form resident language, slang, or regional terms. *Mitigation:* report results honestly as in-distribution; keep preprocessing simple.

**Label leakage columns.** `material_composition` (1 value per category) and `disposal_instruction` (5 per category) are near-perfect proxies for the label. *Mitigation:* the text model uses the `description` column **only**; the other columns serve as RAG reference text, not classifier features.

**Duplicate descriptions.** 61 duplicate texts are removed before splitting to prevent train/test leakage.

**Controlled image capture.** Uniform 524×524 studio-style shots differ from cluttered real phone photos, so the CNN may **not generalise** to deployment images without further data.

**Uneven policy coverage.** Every category has policy text, but **Paper has only 1 document**, so RAG answers for Paper draw on the thinnest evidence.

**Visual similarity.** Paper/Cardboard, Food Organics/Vegetation and the Glass items look alike; expect these as the main CNN confusions and inspect them in Part 2's confusion matrix.